# Feature Engineering: Daily Time Series

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / 'outputs' / 'data' / 'ridership_clean.parquet'
ridership_2023_clean = pd.read_parquet(DATA_PATH)
print(f"{len(ridership_2023_clean):,} trips loaded")

30,830,264 trips loaded


### Build Daily Feature Matrix

Call the full pipeline from `src/feature_engineering`: aggregate, calendar features, lag features, rolling features. Rows with NaN from lag/rolling creation are dropped (we lose the first 28 days).

In [3]:
from src.feature_engineering import build_daily_features

df = build_daily_features(ridership_2023_clean)
print(f"{len(df)} days after dropping NaN rows")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"\nColumns ({len(df.columns)}):")
print(df.columns.tolist())
df.head()

del ridership_2023_clean

2164 days after dropping NaN rows
Date range: 2020-01-29 to 2025-12-31

Columns (26):
['trip_count', 'mean_duration_min', 'median_duration_min', 'pct_annual_member', 'pct_peak_hour', 'day_of_week', 'month', 'day_of_year', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'day_of_year_sin', 'day_of_year_cos', 'is_weekend', 'is_holiday', 'trip_count_lag1', 'trip_count_lag7', 'trip_count_lag14', 'trip_count_lag28', 'trip_count_rmean7', 'trip_count_rstd7', 'trip_count_rmean14', 'trip_count_rstd14', 'trip_count_rmean28', 'trip_count_rstd28']


### Feature Overview

In [4]:
df.describe().round(2)

,trip_count,mean_duration_min,median_duration_min,pct_annual_member,pct_peak_hour,day_of_week,month,day_of_year,day_of_week_sin,day_of_week_cos,...,trip_count_lag1,trip_count_lag7,trip_count_lag14,trip_count_lag28,trip_count_rmean7,trip_count_rstd7,trip_count_rmean14,trip_count_rstd14,trip_count_rmean28,trip_count_rstd28
count,2164.00,2164.00,2164.00,2164.00,2164.00,2164.0,2164.00,2164.00,2164.00,2164.00,...,2164.00,2164.00,2164.00,2164.00,2164.00,2164.00,2164.00,2164.00,2164.00,2164.00
mean,14206.34,13.44,11.37,0.59,0.54,3.0,6.59,185.35,0.00,-0.00,...,14207.00,14210.30,14195.87,14164.31,14208.24,2628.39,14205.41,2827.63,14192.89,3072.80
std,9577.24,1.82,1.90,0.28,0.07,2.0,3.41,104.39,0.71,0.71,...,9576.54,9572.55,9586.06,9613.55,9167.66,1413.17,9098.50,1280.94,9019.31,1242.04
min,155.00,9.85,8.18,0.00,0.29,0.0,1.00,1.00,-0.97,-0.90,...,155.00,155.00,155.00,155.00,778.57,133.09,893.29,286.66,1517.11,510.97
25%,6193.00,11.98,9.92,0.36,0.48,1.0,4.00,95.00,-0.78,-0.90,...,6193.00,6193.00,6156.75,5959.75,6356.25,1598.54,6392.61,1876.93,6413.57,2149.62
50%,12419.50,13.21,11.00,0.69,0.56,3.0,7.00,185.50,0.00,-0.22,...,12419.50,12419.50,12419.50,12419.50,13194.93,2385.92,13451.46,2728.06,13183.77,3162.92
75%,21180.00,14.40,12.17,0.80,0.60,5.0,10.00,276.00,0.78,0.62,...,21180.00,21180.00,21180.00,21180.00,21253.46,3575.35,21066.21,3715.67,21060.72,3885.50
max,44137.00,20.46,19.88,0.97,0.75,6.0,12.00,366.00,0.97,1.00,...,44137.00,44137.00,44137.00,44137.00,38631.43,8543.70,37495.57,7435.36,37262.68,7939.13


In [5]:
# Check for any remaining NaN
nan_counts = df.isna().sum()
if nan_counts.sum() == 0:
    print("There are no missing values. The feature matrix is clean.")
else:
    print("Missing values:")
    print(nan_counts[nan_counts > 0])

There are no missing values. The feature matrix is clean.


In [6]:
# Export to parquet (preserving Trip Id index)
output_path = PROJECT_ROOT / 'outputs' / 'data' / 'ridership.parquet'

# Use snappy compression for a balance of speed and size
# index=True keeps Trip Id in the file for traceability
df.to_parquet(output_path, compression='snappy', index=True)

del df